# 📊 PosePulse — Paper Classification Benchmarks (Kaggle)

Trains **both** classification models on the same ViTPose-S 256-d frame embeddings:

| Model | Architecture | Params |
|---|---|---|
| **BiLSTM-CNN** | BiLSTM(4 hidden/direction) → Conv2D(128 → 256 → 64 → 1) → Linear → Softmax | ~450 K |
| **xLSTM[7:1]** | 7 mLSTM + 1 sLSTM (hidden=256, heads=4) → mean-pool → Linear → Softmax | ~5.2 M |

**Hyperparameters (PosePulse paper §3.3.3):**
- AdamW · lr 3e-4 (cosine to 3e-6 with 10% linear warmup) · betas (0.9, 0.999) · eps 1e-8 · wd 1e-4
- 50 epochs · batch 64 · grad-clip 1.0 · CE loss with label smoothing 0.1
- **EMA decay 0.999** on evaluation weights · best checkpoint by val accuracy

## Setup before running
1. **Add accelerator**: top-right *Settings* → *Accelerator* → choose **GPU T4 ×2** or **GPU P100**.
2. **Upload ViTPose features** as a Kaggle Dataset (call it e.g. `riccio-vit256-features`). It must contain `riccio_realtime_exercise_recognition_biomechanics.npz` and `riccio_realtime_exercise_recognition_labels.npz`. Attach it to this notebook via the right-hand *Input* panel.
3. **Make the project code available** — easiest is to set `GITHUB_URL` in §2 below.

Estimated wall-clock on T4 (both models, 50 epochs, ~10k windows): **30–50 min**.

## 1 · Environment check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv || echo 'No GPU — enable accelerator in Settings.'
import torch, sys, os
print(f'torch {torch.__version__} · cuda available: {torch.cuda.is_available()} · py {sys.version.split()[0]}')
if torch.cuda.is_available():
    print(f'device: {torch.cuda.get_device_name(0)}')
else:
    print('[warn] training will fall back to CPU — much slower.')

## 2 · Pull project code + locate features dataset

In [ ]:
GITHUB_URL = ''   # leave blank: we use a Kaggle dataset instead of git clone

import os, glob, shlex, subprocess, zipfile
from pathlib import Path

WORK = Path('/kaggle/working')
INPUT_ROOT = Path('/kaggle/input')
OUT_ROOT   = WORK / 'paper_classification_vit256'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# --- Diagnostic: show what's actually mounted -------------------------------
print('=== /kaggle/input ===')
if INPUT_ROOT.exists():
    for d in sorted(INPUT_ROOT.iterdir()):
        if d.is_dir():
            print(f'  {d.name}/')
            for x in sorted(d.iterdir())[:10]:
                print(f'    {x.name}{"/" if x.is_dir() else ""}')
else:
    print('  (no /kaggle/input directory — no datasets attached)')
print()

# --- Auto-unzip any source.zip that landed unextracted -----------------------
for z in glob.glob(f'{INPUT_ROOT}/**/*.zip', recursive=True):
    target = Path(WORK) / 'unzipped' / Path(z).stem
    if not target.exists():
        target.mkdir(parents=True, exist_ok=True)
        print(f'[unzip] {z}  →  {target}')
        with zipfile.ZipFile(z) as zf:
            zf.extractall(target)

def _find_project(parents):
    for parent in parents:
        for p in sorted(glob.glob(f'{parent}/**/train_paper_classification.py', recursive=True)):
            return os.path.dirname(p)
    return None

PROJECT_DIR = _find_project([str(INPUT_ROOT), str(WORK)])

if PROJECT_DIR is None and GITHUB_URL:
    os.chdir('/kaggle/working')
    target = Path('/kaggle/working/Finess-coach-capstone')
    if not target.exists():
        subprocess.check_call(['git', 'clone', '--depth=1', GITHUB_URL, str(target)])
    PROJECT_DIR = _find_project([str(WORK)])

assert PROJECT_DIR, (
    'No project found.\n'
    '  Did you attach the posepulse-source dataset to this kernel?\n'
    '  Right panel → Input → + Add Input → search "posepulse-source" → attach.\n'
    '  Then re-run this cell. Searched: /kaggle/input and /kaggle/working.'
)
os.chdir(PROJECT_DIR)
print(f'project → {PROJECT_DIR}')

# Locate the features dataset (must contain *_biomechanics.npz + *_labels.npz).
FEATURE_STEM = 'riccio_realtime_exercise_recognition'
candidates = sorted(glob.glob(f'{INPUT_ROOT}/**/{FEATURE_STEM}_biomechanics.npz', recursive=True))
assert candidates, (
    f'No features file matching {FEATURE_STEM}_biomechanics.npz found under {INPUT_ROOT}. '
    'Attach the riccio-vit256-features dataset via the right-hand Input panel.'
)
FEATURES_DIR = os.path.dirname(candidates[0])
print(f'features → {FEATURES_DIR}')
print(f'out      → {OUT_ROOT}')


## 3 · Install project + verify imports

In [ ]:
import sys, os, subprocess, shutil
from pathlib import Path

# Strategy: copy the project to /kaggle/working/ (writable) and add it to sys.path.
# This sidesteps the read-only /kaggle/input/ mount, which breaks `pip install -e`.
WRITE_ROOT = Path('/kaggle/working/posepulse-src')
if Path(PROJECT_DIR).resolve() != WRITE_ROOT.resolve():
    if WRITE_ROOT.exists():
        shutil.rmtree(WRITE_ROOT)
    shutil.copytree(PROJECT_DIR, WRITE_ROOT)
PROJECT_DIR = str(WRITE_ROOT)
os.chdir(PROJECT_DIR)

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Make sure the subprocess training run also sees the project (it cd's here, but
# explicitly setting PYTHONPATH avoids surprises if cwd handling changes).
os.environ['PYTHONPATH'] = PROJECT_DIR + os.pathsep + os.environ.get('PYTHONPATH', '')

# Try editable install for completeness; fall back silently if it fails (sys.path is enough).
subprocess.call([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', PROJECT_DIR])

from fitness_coach.models.exercise_bilstm_model import ExerciseBiLSTMCNN
from fitness_coach.models.xlstm_model import xLSTMExerciseClassifier
from fitness_coach.datasets.exercise_bilstm_dataset import build_kaggle_frame_feature_datasets
print(f'project → {PROJECT_DIR}')
print('imports OK · BiLSTM-CNN + xLSTM[7:1] + Kaggle loader available')


## 4 · Preflight — verify class counts + window totals before training

In [ ]:
from collections import Counter
from pathlib import Path

tr, va, te, c2i, _, _, _ = build_kaggle_frame_feature_datasets(
    Path(FEATURES_DIR), stem=FEATURE_STEM,
    window=30, stride=15, exclude_coarse_classes=['hammer curl'])

def _per_class(ds, c2i):
    cnt = Counter(int(ds[i][1]) for i in range(len(ds)))
    inv = {i: n for n, i in c2i.items()}
    return {inv[i]: cnt[i] for i in range(len(c2i))}

print(f'classes : {c2i}')
print(f'train   : {len(tr)}  → {_per_class(tr, c2i)}')
print(f'val     : {len(va)}  → {_per_class(va, c2i)}')
print(f'test    : {len(te)}  → {_per_class(te, c2i)}')

## 5 · Train both models

Paper hyperparameters with linear warmup (10%) + cosine decay and EMA-0.999 on eval weights.

In [ ]:
import shlex, subprocess, time

FEATURE_DIM = 256   # 256 = ViTPose-S; 42 if the NPZ was exported with --representation angles

cmd = f'''python -u train_paper_classification.py \
  --kaggle-angles-dir {shlex.quote(FEATURES_DIR)} \
  --kaggle-stem {FEATURE_STEM} \
  --exclude-classes "hammer curl" \
  --feature-dim {FEATURE_DIM} \
  --seq-len 30 --stride 15 --num-classes 4 \
  --bilstm-hidden 4 \
  --xlstm-hidden 256 --xlstm-num-heads 4 --xlstm-block-pattern mmmmmmms \
  --xlstm-conv-kernel-size 4 --xlstm-projection-factor 1.333 \
  --dropout 0.15 \
  --epochs 50 --batch-size 64 --lr 3e-4 --min-lr 3e-6 \
  --weight-decay 1e-4 --grad-clip 1.0 --label-smoothing 0.1 \
  --warmup-frac 0.1 --ema-decay 0.999 \
  --models bilstm xlstm \
  --output-dir {shlex.quote(str(OUT_ROOT))}'''

print(cmd, '\n')
t0 = time.time()
rc = subprocess.call(cmd, shell=True)
print(f'\n[exit={rc}] elapsed={(time.time()-t0)/60:.1f} min')
assert rc == 0, 'Training failed — scroll up for the traceback.'

## 6 · Training curves (both models, side-by-side)

In [ ]:
import json, matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for name, color in [('bilstm_cnn', 'C0'), ('xlstm_7_1', 'C3')]:
    p = OUT_ROOT / name / 'history.json'
    if not p.is_file():
        continue
    h = json.load(open(p))
    ep = [r['epoch'] for r in h]
    ax[0].plot(ep, [r['train_loss'] for r in h], label=name, color=color)
    ax[1].plot(ep, [r['val_acc']    for r in h], label=name, color=color)
ax[0].set_title('train loss');   ax[0].set_xlabel('epoch'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].set_title('val accuracy'); ax[1].set_xlabel('epoch'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_ROOT / 'training_curves.png', dpi=120)
plt.show()
print(f'saved → {OUT_ROOT / "training_curves.png"}')

## 7 · Final results table + bundle

In [ ]:
import json, tarfile
from pathlib import Path

summary = json.load(open(OUT_ROOT / 'summary.json'))
print(f"{'model':<14s}  {'val_acc':>10s}  {'test_acc':>10s}  {'test_f1':>10s}")
print('-' * 52)
for r in summary['runs']:
    print(f"{r['name']:<14s}  {r['best_val_acc']:>10.4f}  {r['test_acc']:>10.4f}  {r['test_f1']:>10.4f}")

bundle = OUT_ROOT / 'paper_classification_bundle.tar.gz'
with tarfile.open(bundle, 'w:gz') as tar:
    for p in Path(OUT_ROOT).rglob('*'):
        if p.is_file() and p.suffix in {'.pt', '.json', '.png'}:
            tar.add(p, arcname=p.relative_to(OUT_ROOT))
print(f'\n📦 bundle → {bundle}  ({bundle.stat().st_size/1e6:.1f} MB)')

---

Deliverables in `/kaggle/working/paper_classification_vit256/`:

- `bilstm_cnn/best.pt`, `bilstm_cnn/history.json`, `bilstm_cnn/metrics.json`
- `xlstm_7_1/best.pt`,  `xlstm_7_1/history.json`,  `xlstm_7_1/metrics.json`
- `summary.json` — combined results across both models
- `training_curves.png` — side-by-side training curves
- `paper_classification_bundle.tar.gz` — everything in one archive

Everything under `/kaggle/working/` persists when you commit the notebook — download the bundle from the *Output* tab on the right.